# Model Experiments & Benchmarking (T-106 & T-107)

This notebook implements and benchmarks baseline and advanced regression models for predicting NYC yellow taxi **fare amount** ($) and **trip duration** (minutes).

### Tickets Covered:
* **T-106 (Baseline Regressors)**: Trivial Mean Baseline & Decision Tree Regressors.
* **T-107 (Gradient Boosted Ensembles)**: LightGBM & XGBoost Regressors.

--- 
### Key Architectural & Design Decisions:
1. **Single-Output vs Multi-Output Models**: We train two separate single-output models (one for `fare_amount`, one for `duration_minutes`) rather than a single multi-output model. Fares ($) and durations (minutes) operate on distinct financial/temporal scales with different error structures.
2. **Reproducible Temporal Splits**: Models are trained on `train_cleaned.parquet` (pre-2022-05-23) and evaluated on `test_cleaned.parquet` (post-2022-05-23).
3. **Feature Leakage Safeguards**: Preprocessing uses the fitted `NYCFeaturePipeline` artifact (`models/feature_pipeline.pkl`) fitted strictly on training data. Feature importances are audited against a 65% dominance threshold.

In [1]:
import os
import sys
import pickle
import time
from pathlib import Path

# Ensure project root directory is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from src.config import (
    TRAIN_CLEANED_PATH,
    TEST_CLEANED_PATH,
    ALLOWED_FEATURES,
    MODELS_DIR,
    RANDOM_SEED,
)
from src.train import (
    calculate_metrics,
    measure_inference_time,
    train_and_evaluate_baselines,
)
from src.gradient_boosting import regression_metrics, measure_single_row_latency

## 1. Feature Engineering & Dataset Preparation
Loads cleaned splits and applies the serialized `NYCFeaturePipeline` artifact (`models/feature_pipeline.pkl`).

In [2]:
print("Loading cleaned dataset splits...")
train_df = pd.read_parquet(TRAIN_CLEANED_PATH)
test_df = pd.read_parquet(TEST_CLEANED_PATH)

pipeline_path = os.path.join(MODELS_DIR, "feature_pipeline.pkl")
with open(pipeline_path, "rb") as f:
    pipeline = pickle.load(f)

X_train_feat = pipeline.transform(train_df[ALLOWED_FEATURES])
X_test_feat = pipeline.transform(test_df[ALLOWED_FEATURES])

y_train_fare, y_test_fare = (
    train_df["fare_amount"].values,
    test_df["fare_amount"].values,
)
y_train_dur, y_test_dur = (
    train_df["duration_minutes"].values,
    test_df["duration_minutes"].values,
)

print(
    f"Engineered feature matrix shape: Train {X_train_feat.shape}, Test {X_test_feat.shape}"
)

Loading cleaned dataset splits...
Engineered feature matrix shape: Train (2402868, 29), Test (899623, 29)


## 2. Baseline Model Evaluation (T-106)
Evaluates Trivial Mean Baseline and Decision Tree Regressor.

In [3]:
baseline_output = train_and_evaluate_baselines(save_models=False)
baseline_summary = baseline_output["summary_table"]
display(baseline_summary)

2026-08-14 14:55:41,856 - INFO - --- Starting Baseline Model Training (T-106) ---
2026-08-14 14:55:41,857 - INFO - Loading cleaned train data from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/dataset/train_cleaned.parquet...
2026-08-14 14:55:41,935 - INFO - Loading cleaned test data from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/dataset/test_cleaned.parquet...
2026-08-14 14:55:41,957 - INFO - Loading existing feature pipeline from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/models/feature_pipeline.pkl...
2026-08-14 14:55:43,673 - INFO - --- Training Trivial Mean Baseline ---
2026-08-14 14:55:43,852 - INFO - --- Training Decision Tree Regressors ---
2026-08-14 14:56:41,342 - INFO - 
=== Baseline Models Evaluation Summary (Test Set) ===
                          MAE     RMSE   MAPE      R2  train_time_sec  inference_latency_ms
Trivial_Mean_Fare      8.8515  13.5709  77.69 -0.0002          0.0700                0.0250
Trivial_Mean_Dur

,MAE,RMSE,MAPE,R2,train_time_sec,inference_latency_ms
Trivial_Mean_Fare,8.8515,13.5709,77.69,-0.0002,0.0700,0.0250
Trivial_Mean_Duration,9.2712,13.4220,99.09,-0.0002,0.0037,0.0224
DecisionTree_Fare,1.4581,2.8822,12.23,0.9549,29.5178,0.8275
DecisionTree_Duration,3.9406,6.3713,30.31,0.7746,26.1798,0.5201


## 3. Gradient Boosted Tree Evaluation (T-107 — LightGBM & XGBoost)
Trains and benchmarks LightGBM and XGBoost models for both target variables.

In [4]:
sample_row = X_test_feat.head(1)
gb_results = {}

# --- 3.1 LightGBM Fare Amount ---
t0 = time.time()
lgb_fare = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    random_state=RANDOM_SEED,
    verbosity=-1,
    n_jobs=-1,
)
lgb_fare.fit(X_train_feat, y_train_fare)
t_lgb_fare = time.time() - t0
p_lgb_fare = lgb_fare.predict(X_test_feat)
m_lgb_fare = calculate_metrics(y_test_fare, p_lgb_fare)
lat_lgb_fare = measure_inference_time(lgb_fare, sample_row)
gb_results["LightGBM_Fare"] = {
    **m_lgb_fare,
    "train_time_sec": round(t_lgb_fare, 4),
    "inference_latency_ms": lat_lgb_fare,
}

# --- 3.2 LightGBM Duration Minutes ---
t0 = time.time()
lgb_dur = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    random_state=RANDOM_SEED,
    verbosity=-1,
    n_jobs=-1,
)
lgb_dur.fit(X_train_feat, y_train_dur)
t_lgb_dur = time.time() - t0
p_lgb_dur = lgb_dur.predict(X_test_feat)
m_lgb_dur = calculate_metrics(y_test_dur, p_lgb_dur)
lat_lgb_dur = measure_inference_time(lgb_dur, sample_row)
gb_results["LightGBM_Duration"] = {
    **m_lgb_dur,
    "train_time_sec": round(t_lgb_dur, 4),
    "inference_latency_ms": lat_lgb_dur,
}

# --- 3.3 XGBoost Fare Amount ---
t0 = time.time()
xgb_fare = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=RANDOM_SEED,
    tree_method="hist",
    n_jobs=-1,
)
xgb_fare.fit(X_train_feat, y_train_fare)
t_xgb_fare = time.time() - t0
p_xgb_fare = xgb_fare.predict(X_test_feat)
m_xgb_fare = calculate_metrics(y_test_fare, p_xgb_fare)
lat_xgb_fare = measure_inference_time(xgb_fare, sample_row)
gb_results["XGBoost_Fare"] = {
    **m_xgb_fare,
    "train_time_sec": round(t_xgb_fare, 4),
    "inference_latency_ms": lat_xgb_fare,
}

# --- 3.4 XGBoost Duration Minutes ---
t0 = time.time()
xgb_dur = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=RANDOM_SEED,
    tree_method="hist",
    n_jobs=-1,
)
xgb_dur.fit(X_train_feat, y_train_dur)
t_xgb_dur = time.time() - t0
p_xgb_dur = xgb_dur.predict(X_test_feat)
m_xgb_dur = calculate_metrics(y_test_dur, p_xgb_dur)
lat_xgb_dur = measure_inference_time(xgb_dur, sample_row)
gb_results["XGBoost_Duration"] = {
    **m_xgb_dur,
    "train_time_sec": round(t_xgb_dur, 4),
    "inference_latency_ms": lat_xgb_dur,
}

gb_summary = pd.DataFrame(gb_results).T
display(gb_summary)

,MAE,RMSE,MAPE,R2,train_time_sec,inference_latency_ms
LightGBM_Fare,1.4214,2.9736,12.13,0.9520,24.4735,0.7459
LightGBM_Duration,3.7608,6.0491,29.50,0.7968,18.7628,0.7742
XGBoost_Fare,1.4452,3.0496,12.32,0.9495,18.7765,3.3908
XGBoost_Duration,3.8350,6.0984,30.70,0.7935,-31.9619,2.5182


## 4. Feature Importance & Leakage Audit
Checks feature importances across GBDT models to verify no feature exceeds the 65% dominance leakage threshold.

In [5]:
feature_names = list(X_train_feat.columns)

# LightGBM Feature Importances
lgb_fare_imp = pd.Series(
    lgb_fare.feature_importances_, index=feature_names
).sort_values(ascending=False)
lgb_dur_imp = pd.Series(lgb_dur.feature_importances_, index=feature_names).sort_values(
    ascending=False
)

print("=== LightGBM Top 10 Features (Fare Amount) ===")
display(lgb_fare_imp.head(10))

print("\n=== LightGBM Top 10 Features (Trip Duration) ===")
display(lgb_dur_imp.head(10))

# Check Leakage Threshold (Max Importance < 65%)
max_lgb_fare = lgb_fare_imp.max() / lgb_fare_imp.sum()
max_lgb_dur = lgb_dur_imp.max() / lgb_dur_imp.sum()
print(f"\nLeakage Audit Check:")
print(f"- Max normalized feature importance (LightGBM Fare): {max_lgb_fare * 100:.2f}%")
print(
    f"- Max normalized feature importance (LightGBM Duration): {max_lgb_dur * 100:.2f}%"
)
assert max_lgb_fare < 0.65, "Feature dominance threshold exceeded for LightGBM Fare"
assert max_lgb_dur < 0.65, "Feature dominance threshold exceeded for LightGBM Duration"
print("STATUS: PASSED (No suspicious feature leakage detected).")

=== LightGBM Top 10 Features (Fare Amount) ===


trip_distance              2468
RatecodeID                 1725
pickup_hour                1278
do_lon                     1189
pu_lat                     1176
pu_lon                     1091
do_lat                     1084
cos_hour                   1071
pickup_day                  984
PULocationID_target_enc     771
dtype: int32


=== LightGBM Top 10 Features (Trip Duration) ===


trip_distance       1926
do_lat              1850
do_lon              1618
pu_lat              1328
pickup_hour         1294
pu_lon              1253
pickup_day          1143
pickup_dayofweek    1025
cos_hour             970
haversine_ratio      792
dtype: int32


Leakage Audit Check:
- Max normalized feature importance (LightGBM Fare): 13.27%
- Max normalized feature importance (LightGBM Duration): 10.35%
STATUS: PASSED (No suspicious feature leakage detected).


## 5. Master Model Performance Leaderboard
Combined comparison table of all evaluated baseline and GBDT models on the unseen temporal test set.

In [6]:
all_results = {**baseline_output["results"], **gb_results}
master_df = pd.DataFrame(all_results).T
print("=== Master Leaderboard (All Models on Test Set) ===")
display(master_df)

=== Master Leaderboard (All Models on Test Set) ===


,MAE,RMSE,MAPE,R2,train_time_sec,inference_latency_ms
Trivial_Mean_Fare,8.8515,13.5709,77.69,-0.0002,0.0700,0.0250
Trivial_Mean_Duration,9.2712,13.4220,99.09,-0.0002,0.0037,0.0224
DecisionTree_Fare,1.4581,2.8822,12.23,0.9549,29.5178,0.8275
DecisionTree_Duration,3.9406,6.3713,30.31,0.7746,26.1798,0.5201
LightGBM_Fare,1.4214,2.9736,12.13,0.9520,24.4735,0.7459
LightGBM_Duration,3.7608,6.0491,29.50,0.7968,18.7628,0.7742
XGBoost_Fare,1.4452,3.0496,12.32,0.9495,18.7765,3.3908
XGBoost_Duration,3.8350,6.0984,30.70,0.7935,-31.9619,2.5182
